# TRAIN V3 DISK-SAFE — GROUP C1
Train HCM0421 on GPU 0 and HCM0540 on GPU 1. Attach 01_BASE_CODE.zip, 02_ROUND2_DATA.zip, and 03_ROUND2_PATCH_V3_DISKSAFE.zip. No new optimizer checkpoints are written.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'CUDA is not available'
assert torch.cuda.device_count() >= 2, 'Select GPU T4 x2'
print('GPU count:', torch.cuda.device_count())

In [ ]:
from pathlib import Path
import os, shutil
input_root = Path('/kaggle/input')
work_dir = Path('/kaggle/working/vai_nvs')
if work_dir.exists(): shutil.rmtree(work_dir)
base_zips = list(input_root.rglob('01_BASE_CODE.zip'))
if not base_zips: base_zips = list(input_root.rglob('vai_nvs_kaggle_prod_v3.zip'))
if base_zips:
    assert len(base_zips) == 1, f'Expected one BASE CODE ZIP: {base_zips}'
    shutil.unpack_archive(str(base_zips[0]), str(work_dir))
else:
    candidates = []
    for gs_dir in input_root.rglob('gaussian-splatting'):
        parent = gs_dir.parent
        complete = (parent/'scripts/setup_cuda_extensions_linux.sh').exists() and (gs_dir/'submodules/diff-gaussian-rasterization').exists()
        if complete: candidates.append(parent)
    assert len(candidates) == 1, f'BASE CODE missing or duplicated: {candidates}'
    shutil.copytree(candidates[0], work_dir)
assert (work_dir/'scripts/setup_cuda_extensions_linux.sh').exists(), 'Incomplete BASE CODE'
assert (work_dir/'gaussian-splatting/submodules/diff-gaussian-rasterization').exists(), 'CUDA submodule missing'
os.chdir(work_dir)
print('Workspace:', Path.cwd())

In [ ]:
!python -m pip install -U pip "setuptools<82" wheel ninja
!pip install tqdm plyfile opencv-python pillow scipy matplotlib
from pathlib import Path
import shutil
markers = list(Path('/kaggle/input').rglob('ROUND2_PATCH_V3_DISKSAFE.txt'))
if not markers:
    patch_zips = list(Path('/kaggle/input').rglob('03_ROUND2_PATCH_V3_DISKSAFE.zip'))
    assert len(patch_zips) == 1, f'Add exactly one V3 disk-safe patch: {patch_zips}'
    patch_root = Path('/kaggle/working/round2_patch')
    if patch_root.exists(): shutil.rmtree(patch_root)
    shutil.unpack_archive(str(patch_zips[0]), str(patch_root))
    markers = list(patch_root.rglob('ROUND2_PATCH_V3_DISKSAFE.txt'))
assert len(markers) == 1, f'Patch marker problem: {markers}'
patch_root = markers[0].parent
patches = ['gaussian-splatting/scene/cameras.py','gaussian-splatting/train.py','gaussian-splatting/render.py','scripts/train_round2_group_2gpu.sh','scripts/train_round2_v2_2gpu.sh','tools/prepare_round2_kaggle.py','tools/undistort_simple_radial_dataset.py','tools/prune_colmap_images_to_existing.py','tools/colmap_io.py']
for relative in patches:
    source, target = patch_root/relative, Path('/kaggle/working/vai_nvs')/relative
    assert source.exists(), f'Missing patch file: {source}'
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)
print('V3 disk-safe patch applied')

In [ ]:
!python tools/prepare_round2_kaggle.py --group C1
import os, torch
major, minor = torch.cuda.get_device_capability(0)
os.environ['TORCH_CUDA_ARCH_LIST'] = f'{major}.{minor}'
!bash scripts/setup_cuda_extensions_linux.sh

In [ ]:
import os, re, shutil, subprocess
from pathlib import Path
scenes = ['HCM0421', 'HCM0540']
subprocess.run(['python','-m','pip','cache','purge'], check=False)
for scene in scenes:
    model_dir = Path(f'outputs/prod_private_{scene}')
    checkpoints = []
    for candidate in Path('/kaggle/input').rglob(f'prod_private_{scene}/chkpnt*.pth'):
        match = re.fullmatch(r'chkpnt(\d+)\.pth', candidate.name)
        if match and int(match.group(1)) < 40000: checkpoints.append((int(match.group(1)), candidate))
    if checkpoints:
        iteration, selected = max(checkpoints, key=lambda item: item[0])
        model_dir.mkdir(parents=True, exist_ok=True)
        (model_dir/selected.name).symlink_to(selected)
        print('RESUME WITHOUT COPYING', scene, iteration, selected)
env = os.environ.copy()
env.update(ROUND2_GROUP='C1', V2_ITERATIONS='40000', V2_ENABLE_CHECKPOINTS='0', V2_MIN_DISK_GIB='6')
subprocess.run(['bash','scripts/train_round2_v2_2gpu.sh'], check=True, env=env)

In [ ]:
from pathlib import Path
for scene in ['HCM0421','HCM0540']:
    ply = Path(f'outputs/prod_private_{scene}/point_cloud/iteration_40000/point_cloud.ply')
    assert ply.exists(), f'Missing final model: {scene}'
    print(scene, 'OK', round(ply.stat().st_size/1024**2, 1), 'MiB')
print('GROUP C1 VERIFIED — SAVE VERSION')